# KWISMO — Notebook 01 : Analyse Exploratoire des Données (EDA)

Ce notebook effectue l'analyse statistique et visuelle du jeu de données KWISMO.

### Compatibilité Multi-Sources & Environnements :
- Fonctionne en **Local**, sur **Google Colab** et sur **Kaggle Notebooks**.
- Supporte aussi bien les fichiers de données brutes (`messages.jsonl`) que les fichiers nettoyés (`model_b_clean.jsonl` / `model_b_augmented.jsonl`).

In [ ]:
# 1. Connexion Google Drive (Colab), Détection d'environnement & Localisation du Dataset
import os
import sys
import json
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

# Sur Colab : monter Google Drive automatiquement si pas encore monté
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive
        print("Connexion automatique à Google Drive...")
        drive.mount('/content/drive')
    except Exception as err:
        print(f"Montage manuel recommandé : {err}")

# Auto-détection de la racine du projet kwismo-ai
current_dir = Path.cwd()
repo_root = current_dir
for candidate in [current_dir, current_dir.parent, current_dir.parent.parent, Path('/content/kwismo/kwismo-ai'), Path('/kaggle/working/kwismo/kwismo-ai')]:
    if (candidate / "data").exists() or (candidate / "src").exists():
        repo_root = candidate
        break

print(f"Répertoire courant : {current_dir}")
print(f"Racine projet détectée : {repo_root}\n")

# Emplacements candidats exhaustifs (y compris à la racine du Google Drive)
POSSIBLE_PATHS = [
    Path("/content/drive/MyDrive/messages.jsonl"),
    Path("/content/drive/MyDrive/kwismo_data/messages.jsonl"),
    Path("/content/drive/MyDrive/kwismo/messages.jsonl"),
    Path("/content/drive/MyDrive/Kwismo/messages.jsonl"),
    repo_root / "data" / "processed" / "model_b_augmented.jsonl",
    repo_root / "data" / "processed" / "model_b_clean.jsonl",
    repo_root / "data" / "raw" / "kwismo_data" / "messages.jsonl",
    repo_root / "data" / "raw" / "scraped" / "messages.jsonl",
    Path("/content/kwismo/kwismo-ai/data/processed/model_b_augmented.jsonl"),
    Path("/content/kwismo/kwismo-ai/data/processed/model_b_clean.jsonl"),
    Path("/kaggle/working/kwismo/kwismo-ai/data/processed/model_b_augmented.jsonl"),
    Path("/kaggle/working/kwismo/kwismo-ai/data/processed/model_b_clean.jsonl")
]

print("=== Rapport de diagnostic des emplacements du dataset ===")
dataset_path = None
for p in POSSIBLE_PATHS:
    exists = p.exists()
    status = "[TROUVÉ]" if exists else "[ABSENT]"
    print(f"{status} {p}")
    if exists and dataset_path is None:
        dataset_path = p

# Si aucun chemin prédéfini n'a fonctionné, recherche récursive dans Drive et Colab/Kaggle
if dataset_path is None:
    print("\nRecherche récursive dans Google Drive et répertoires système...")
    search_roots = [Path('/content/drive/MyDrive'), repo_root, Path('/content'), Path('/kaggle/working')]
    for search_root in search_roots:
        if search_root.exists():
            found_files = list(search_root.rglob("*.jsonl"))
            for ff in found_files:
                if "checkpoint" not in str(ff):
                    dataset_path = ff
                    print(f"Fichier de données trouvé dynamiquement : {dataset_path}")
                    break
        if dataset_path:
            break

if dataset_path:
    print(f"\nDataset sélectionné avec succès : {dataset_path}")
else:
    error_msg = (
        "\nERREUR : Le fichier messages.jsonl n'est pas accessible par Colab.\n" 
        "Vérifiez que vous avez autorisé Colab à accéder à votre Google Drive en exécutant :\n"
        "  from google.colab import drive; drive.mount('/content/drive')\n"
    )
    raise FileNotFoundError(error_msg)

## 2. Chargement et Statistiques Globales

In [ ]:
import pandas as pd
import numpy as np

records = []
if dataset_path and dataset_path.is_file():
    with open(dataset_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Nombre total de lignes chargées : {len(df)}")
print("Colonnes présentées dans le jeu de données :", list(df.columns))
if not df.empty:
    display(df.head(5))

## 3. Analyse de la Longueur des Textes (Mots & Caractères)

In [ ]:
text_col = None
for c in ["texte", "text", "content", "snippet", "description"]:
    if c in df.columns:
        text_col = c
        break

if not df.empty and text_col:
    df["longueur_caracteres"] = df[text_col].fillna("").apply(len)
    df["nombre_mots"] = df[text_col].fillna("").apply(lambda t: len(t.split()))
    
    print(f"=== Statistiques sur le nombre de mots (Colonne : '{text_col}') ===")
    print(df["nombre_mots"].describe())
    
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 4))
    plt.hist(df["nombre_mots"], bins=30, color="#2FAC66", edgecolor="black")
    plt.title(f"Distribution du nombre de mots par extrait de fraude (colonne: {text_col})")
    plt.xlabel("Nombre de mots")
    plt.ylabel("Fréquence")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.show()
else:
    print("Aucune colonne de texte explicite trouvée pour l'analyse de longueur.")

## 4. Analyse des Mots-Clés Récurrents (N-grams & Termes de Fraude)

In [ ]:
from collections import Counter
import re

if not df.empty and text_col:
    all_words = []
    stopwords = {"de", "la", "le", "les", "des", "un", "une", "et", "a", "en", "du", "pour", "sur", "est", "pas", "plus", "par", "que", "dans", "avec", "au", "ce", "qui", "ne", "https", "http", "com"}
    
    for text in df[text_col].dropna():
        words = re.findall(r"\b\w+\b", str(text).lower())
        filtered = [w for w in words if w not in stopwords and len(w) > 2 and not w.isdigit()]
        all_words.extend(filtered)
        
    counter = Counter(all_words)
    print("=== Top 20 des mots les plus fréquents ===")
    top20 = counter.most_common(20)
    for word, freq in top20:
        print(f"{word:20s} : {freq}")
else:
    print("Aucune colonne de texte disponible pour l'analyse par N-grams.")

## 5. Répartition par Type d'Origine et Source

In [ ]:
# 5. Répartition par Type d'Origine et Source (Flexible pour datasets bruts et nettoyés)
if not df.empty:
    print("=== Répartition par Type d'Origine (texte / image_ocr) ===")
    type_col = None
    for c in ["type_origine", "type", "content_type", "media_type"]:
        if c in df.columns:
            type_col = c
            break
    
    if type_col:
        print(f"Colonne utilisée : '{type_col}'")
        print(df[type_col].value_counts())
    else:
        print("ℹAucune colonne de type spécifique trouvée (toutes les entrées sont considérées comme du texte).")

    print("\n=== Répartition par Source / Provenance ===")
    source_col = None
    for c in ["source_origine", "source", "url", "domain", "domaine", "site"]:
        if c in df.columns:
            source_col = c
            break

    if source_col:
        print(f"Colonne utilisée : '{source_col}'")
        print(df[source_col].value_counts().head(10))
    else:
        print("ℹAucune colonne de source/provenance explicite présente dans ce fichier de données.")
else:
    print("Le DataFrame est vide.")